# LoRA reproduction on Colab

Runtime > Change runtime type > T4 GPU (free tier is enough for RoBERTa-base on MRPC).

Run the cells in order. Only cell 3 needs editing: point it at your repo.

## 1. Confirm the GPU

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Wed Aug 19 10:32:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Dependencies\n\nColab ships with torch and a recent transformers, so this is usually a no-op plus `datasets`.

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.18" pytest

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

torch 2.11.0+cu128 | cuda True


## 3. Get the code

Pick one. Cloning is better once the repo exists, since it keeps Colab and your
laptop on the same commit. Uploading a zip is fine for the first session.

In [ ]:
REPO_URL = "https://github.com/Hani0101/lora-fairness.git"
PROJECT_DIR = "/content/lora-fairness"

import os, shutil
from pathlib import Path

if REPO_URL:
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    !git clone -q $REPO_URL $PROJECT_DIR
else:
    # Fallback: upload lora-fairness.zip when prompted.
    from google.colab import files
    uploaded = files.upload()
    archive = next(iter(uploaded))
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    shutil.unpack_archive(archive, "/content")

os.chdir(PROJECT_DIR)
print(sorted(p.name for p in Path(".").iterdir()))

['.git', 'README.md', 'notebooks', 'requirements.txt', 'src', 'tests']


## 4. Run the tests first\n\nIf these fail, nothing below is worth running.

In [ ]:
!python -m pytest tests -q

....................                                                     [100%]
20 passed in 1.87s


## 5. Smoke run

200 examples, 1 epoch. Proves the data download, tokenizer, injection, and
training loop all work before you spend GPU time on the real sweep.

In [ ]:
!python -m src.train --method lora --task mrpc --model roberta-base \
    --train-subset 200 --epochs 1 --log-every 5

config.json: 100% 481/481 [00:00<00:00, 1.99MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 154kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 1.59MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 9.51MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 2.70MB/s]
README.md: 100% 35.3k/35.3k [00:00<00:00, 64.5MB/s]

mrpc/train-00000-of-00001.parquet: downloading bytes:  92% 596k/649k [00:01<00:00, 446kB/s]
mrpc/train-00000-of-00001.parquet: downloading bytes: 100% 646k/646k [00:01<00:00, 472kB/s, 62.7kB/s  ]
mrpc/train-00000-of-00001.parquet: reconstructing file: 100% 649k/649k [00:01<00:00, 474kB/s, 63.2kB/s  ]

mrpc/validation-00000-of-00001.parquet: downloading bytes:   0% 0.00/75.7k [00:00<?, ?B/s]
mrpc/validation-00000-of-00001.parquet: downloading bytes: 100% 74.3k/74.3k [00:01<00:00, 57.7kB/s, 7.23kB/s  ]
mrpc/validation-00000-of-00001.parquet: reconstructing file: 100% 75.7k/75.7k [00:01<00:00, 58.8kB/s, 7.36kB/s  ]

mrpc/test-00000-of-00001.parquet: downloading bytes:   0

## 6. The comparison sweep

Two methods x three seeds. Change `MODEL` to `distilbert-base-uncased` if you
want each run to take about a third as long.

In [9]:
MODEL = "roberta-base"
TASK = "mrpc"
SEEDS = [0, 1, 2]
METHODS = ["lora", "full"]
EPOCHS = 10

import itertools, time, subprocess

for method, seed in itertools.product(METHODS, SEEDS):
    print(f"\n=== {method} seed={seed} ===", flush=True)
    start = time.time()
    subprocess.run([
        "python", "-m", "src.train",
        "--method", method,
        "--task", TASK,
        "--model", MODEL,
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--log-every", "0",
    ], check=True)
    print(f"{method} seed={seed} took {time.time() - start:.0f}s", flush=True)


=== lora seed=0 ===
lora seed=0 took 364s

=== lora seed=1 ===
lora seed=1 took 376s

=== lora seed=2 ===
lora seed=2 took 378s

=== full seed=0 ===
full seed=0 took 601s

=== full seed=1 ===
full seed=1 took 599s

=== full seed=2 ===
full seed=2 took 599s


## 7. Collect the results table

In [10]:
import json
import pandas as pd
from pathlib import Path

rows = []
for path in sorted(Path("runs").glob("*/results.json")):
    r = json.loads(path.read_text())
    rows.append({
        "method": r["args"]["method"],
        "seed": r["args"]["seed"],
        "model": r["args"]["model"],
        "trainable": r["params"]["trainable"],
        "lora_only": r["params"]["lora_only"],
        "trainable_pct": r["params"]["trainable_pct"],
        "best_epoch": r["best"]["epoch"],
        **{k: round(v, 4) for k, v in r["best"].items() if k != "epoch"},
        "sec": r["wall_clock_sec"],
    })

df = pd.DataFrame(rows)
display(df)
display(df.groupby("method")[["accuracy", "f1", "trainable"]].agg(["mean", "std"]))

,method,seed,model,trainable,lora_only,trainable_pct,best_epoch,accuracy,f1,sec
0,full,0,roberta-base,124647170,0,100.00,3,0.8873,0.9187,582.7
1,full,1,roberta-base,124647170,0,100.00,4,0.8897,0.9215,582.2
2,full,2,roberta-base,124647170,0,100.00,9,0.8897,0.9215,582.3
3,lora,0,roberta-base,887042,294912,0.71,6,0.8971,0.9258,344.9
4,lora,1,roberta-base,887042,294912,0.71,8,0.8799,0.9160,357.9
5,lora,2,roberta-base,887042,294912,0.71,5,0.8701,0.9042,360.1
6,lora,42,roberta-base,887042,294912,0.71,1,0.6838,0.8122,4.1


accuracy                  f1              trainable     
            mean       std      mean       std         mean  std
method                                                          
full    0.888900  0.001386  0.920567  0.001617  124647170.0  0.0
lora    0.832725  0.099909  0.889550  0.052317     887042.0  0.0

In [11]:
import shutil
from google.colab import files

shutil.make_archive("mrpc_results", "zip", "runs")
files.download("mrpc_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Keeping the session alive

Free Colab disconnects on idle and caps sessions at roughly 12 hours. The full
sweep above is well under an hour, so this mostly matters if you leave the tab
in the background. Keep the tab open and visible while it runs.

If a run dies partway, results already written to Drive are safe. Re-run cell 7
with `SEEDS` trimmed to whatever is missing.